In [ ]:
import sys
from pathlib import Path

_SRC = Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from fraud_detect import config, data, features, viz

warnings.filterwarnings("ignore")
viz.configure_style()

## Context 

raw features capture the full predictive signal in the data. FE transfroms domain knowledge into model inputs. creating new perspectives on the data that can imporve model performance significanly

## objective 

- create time-derived features from TransactionDT
- build card-level based on address level aggregation features
- engginer email domain and interaction features
-  document transformation logic for profuction reproducibility

In [ ]:
# load data

train = pd.read_parquet(config.MERGED_TRAIN_PATH)
print(f'Data loaded: {train.shape}')

In [ ]:
# time features

train = features.add_time_features(train)

print('Time features created:')
print(train[['hour', 'day_of_week', 'is_night', 'is_weekend']].describe().transpose())

## insight : time based feature engginering

from the result we have created these essential time features:
- hour : hour of day 00-23. fraud rates peak during night hours
- day_of_week : 0= monday to 6 = sunday. weekend patterns differ from weekdays
- is_night : binary flag for 10 P.M to 5 A.M transaction when fraud is elevated
- is_weekend : binary flag for saturday and sunday transactions

these simple features consistenly rank in the top 50 by importance in tree models, providing high value for low engginering cost

In [ ]:
# AMOUNT FEATURES

train = features.add_amount_features(train)

print('Amount Features:')
print(train[['amt_log', 'amt_decimal', 'amt_is_round']].describe())

## insight amount based feature engginering

from the result we created amount based signals :

- TransactionAmt_log : handles the extreme right skew in amount distribution
- TransactionAmt_decimal : extract the decimal portion to detect .00 abd .99 pattern
- is_round_amount : binary flah for whole dollar amounts which may indicate bot behavior

Fraud patterns : very small amounts ( card testing) and round amounts ( automated transaction) correlate with higher fraud rates.

In [ ]:
# card aggregations

train = features.add_card_aggregations(train)

print('Card aggregation features:')
print(train[['card1_amt_mean', 'card1_amt_std', 'card1_tx_count', 'amt_vs_card_mean']].describe())

The core fraud detection question: *Is this transaction consistent with historical behavior?*

- **card1_amt_mean/std**: User's spending baseline and consistency
- **card1_count**: New cards (low count) have higher fraud risk
- **amt_vs_card_mean**: $500 for a user with $50 average is a red flag; same amount for a $2000 average user is normal

In [ ]:
# email features

train = features.add_email_features(train)

print(f"email match rate: {train['email_match'].mean()*100:.1f}%")
print(f"P_email_is_free rate: {train['p_email_is_free'].mean()*100:.1f}%")
print(f"R_email_is_free rate: {train['r_email_is_free'].mean()*100:.1f}%")

## email domains encode trust level : 
- email_match = purcharser/recipient mismatch indicates elevated risk
- P/R_email_is_free : free email are easy to create in bulk. corporate emails indicate higher trust
- combined signal : both free email + domain missmatch = highest fraud probability

In [ ]:
# save features

train.to_parquet(config.PROCESSED_TRAIN_PATH, engine="pyarrow", compression="snappy")
print(f'Saved to: {config.PROCESSED_TRAIN_PATH}')
print(f'Final shape: {train.shape}')